# 03 · Clustering: Gower distance + hierarchical clustering

**Research question:** What health, lifestyle, and environmental patterns are associated with children's screen-time levels?

**Method (approved):** Gower dissimilarity on 14 mixed-type features → hierarchical clustering (average linkage; complete linkage as comparison) on a stratified sample of 10,000 children → silhouette analysis → assignment of the remaining children → stability and sensitivity checks → cluster profiling.

**Status of this notebook:** Sections 0–6 only (setup, data validation, feature specification, Gower implementation and validation, sampling, and a 2,000-child **pilot** that measures runtime and memory). The full 10,000-child clustering (Sections 7+) is added only after the pilot has been reviewed.

> Screen time is a clustering *feature*, not a prediction target. Results describe associations in the survey sample, not causal effects.

## 0 · Setup and configuration

All tunable settings are defined once here so that every later step is reproducible.

- `SEED = 42` is used for every random operation (sampling, row shuffling).
- Memory is monitored for every heavy step: wall-clock time, the peak memory allocated *during* the step (`tracemalloc`, which tracks NumPy arrays), and the process's resident memory (RSS) before and after.
- `require_memory()` stops a step **before** a large allocation if the machine does not have enough free RAM, instead of letting Windows swap to disk.

In [1]:
import gc
import platform
import time
import tracemalloc
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import scipy
import sklearn
from scipy.cluster.hierarchy import cophenet, fcluster, is_monotonic, is_valid_linkage, linkage
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_samples

# ---- Configuration -------------------------------------------------------
SEED = 42
SAMPLE_SIZE = 10_000              # children per replicate sample
N_PARTS = 5                       # disjoint stratified replicate samples (part 0 = primary)
PILOT_SIZE = 2_000                # real-data pilot (subsample of part 0)
STRATA_COLS = ["formtype", "screentime"]   # 3 x 5 = 15 strata
K_RANGE = list(range(2, 11))      # candidate numbers of clusters
LINKAGE_METHODS = ["average", "complete"]  # Ward/centroid/median excluded: they require Euclidean geometry
GOWER_BLOCK_ROWS = 500            # rows per block when building the distance matrix
ASSIGN_CHUNK_ROWS = 1_000         # rows per chunk when assigning non-sample children
AMBIGUITY_THRESHOLDS = (0.005, 0.01, 0.02)  # assignment-margin thresholds to compare (Gower units)
MIN_CLUSTER_SHARE = 0.01          # clusters < 1% of the sample are flagged and investigated (not discarded)
SILHOUETTE_GUIDELINE = 0.02       # practical comparison guideline, not a statistical threshold

DATA_PATH = Path.cwd() / "datasets" / "preprocessed_data.csv"
sklearn.set_config(working_memory=256)   # cap scikit-learn chunked computations at 256 MiB

print("Python  ", platform.python_version())
for mod in (np, pd, scipy, sklearn, psutil):
    print(f"{mod.__name__:8s}", mod.__version__)
vm = psutil.virtual_memory()
print(f"\nRAM total {vm.total / 2**30:.1f} GiB | available now {vm.available / 2**30:.2f} GiB | logical CPUs {psutil.cpu_count()}")

Python   3.12.7
numpy    1.26.4
pandas   2.2.2
scipy    1.13.1
sklearn  1.5.1
psutil   5.9.0

RAM total 13.8 GiB | available now 1.39 GiB | logical CPUs 16


In [2]:
# ---- Memory and runtime monitoring helpers --------------------------------
PROC = psutil.Process()
PERF_LOG = []          # one row per monitored step


def rss_mib():
    return PROC.memory_info().rss / 2**20


def available_mib():
    return psutil.virtual_memory().available / 2**20


def require_memory(n_bytes, label, factor=1.5):
    # Stop before allocating if free RAM is below factor x the expected requirement.
    need_mib = n_bytes * factor / 2**20
    if need_mib > available_mib():
        raise MemoryError(
            f"[{label}] needs ~{need_mib:,.0f} MiB (incl. x{factor} margin) but only "
            f"{available_mib():,.0f} MiB is free. Close other applications and re-run."
        )


@contextmanager
def monitor(step, **info):
    # Record wall time, peak traced allocation, and RSS for one step.
    gc.collect()
    rss_before = rss_mib()
    tracemalloc.start()
    t0 = time.perf_counter()
    try:
        yield
    finally:
        seconds = time.perf_counter() - t0
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        PERF_LOG.append({
            "step": step, **info,
            "seconds": round(seconds, 3),
            "peak_alloc_mib": round(peak / 2**20, 1),
            "rss_before_mib": round(rss_before, 1),
            "rss_after_mib": round(rss_mib(), 1),
            "available_after_mib": round(available_mib(), 0),
        })

## 1 · Load and validate the processed dataset

**What:** load `datasets/preprocessed_data.csv` (exported by `01_preprocessing.ipynb`) and check its schema.

**Why:** every later step assumes a fixed set of columns, a unique identifier, and no ordinary missing values in the clustering features.

**Checks:**
- 51,375 rows; all expected column groups present.
- `hhid` is unique and complete (the stable child identifier; it is never a feature).
- The 15 processed variables have no missing values.
- `sleep_below_minimum` is missing **only** for infants under 4 months (`sc_age_lt4 = 1`). This is *structural* missingness: no sleep recommendation applies at that age. It is not nonresponse, so it is not imputed and does not make a child "incomplete".

In [3]:
df = pd.read_csv(DATA_PATH)

ID_COL = "hhid"
PROCESSED_VARS = ["sc_age_years", "sc_sex", "family_r", "blindness", "eyedoctor", "k4q22_r", "k4q23",
                  "screentime", "sleep_hours", "k6q71_r", "higrade", "foodsit", "k10q12", "k10q13", "k10q14"]
METADATA_COLS = ["formtype", "fwc", "fipsst", "stratum", "sc_age_lt4"]
DERIVED_COLS = ["sleep_below_minimum"]
FLAG_COLS = [c for c in df.columns if c.endswith("_imputed")]

expected = [ID_COL] + PROCESSED_VARS + METADATA_COLS + DERIVED_COLS
missing_cols = sorted(set(expected) - set(df.columns))
assert not missing_cols, f"Missing columns: {missing_cols}"
assert len(df) == 51_375, len(df)
assert df[ID_COL].is_unique and df[ID_COL].notna().all()
assert df[PROCESSED_VARS].notna().all().all()

# Structural missingness: sleep_below_minimum is NaN exactly for infants < 4 months
sleep_na = df["sleep_below_minimum"].isna()
under_4m = df["sc_age_lt4"].eq(1)
assert sleep_na.equals(under_4m), "sleep_below_minimum missingness does not match sc_age_lt4"
assert df.loc[under_4m, "sc_age_years"].eq(0).all()

print("Shape:", df.shape)
print("Imputation flags:", FLAG_COLS)
print(f"Structural missing in sleep_below_minimum (infants < 4 months): {int(sleep_na.sum())}")
print(df[["formtype"]].value_counts().sort_index().to_string())

Shape: (51375, 34)
Imputation flags: ['family_r_imputed', 'blindness_imputed', 'eyedoctor_imputed', 'k4q22_r_imputed', 'k4q23_imputed', 'screentime_imputed', 'sleep_hours_imputed', 'k6q71_r_imputed', 'foodsit_imputed', 'k10q12_imputed', 'k10q13_imputed', 'k10q14_imputed']
Structural missing in sleep_below_minimum (infants < 4 months): 446
formtype
T1          13841
T2          15574
T3          21960


## 2 · Feature specification (approved 14-feature model)

Each feature's **Gower type** determines how two children are compared on it:

| Type | Per-feature dissimilarity `d` | Used for |
|---|---|---|
| `numeric` / `ordinal` | `|x_i − x_j| / range`, using the **fixed codebook range** (ordinal codes used as ranks; Kaufman & Rousseeuw) | age; ordered response scales |
| `nominal` | 0 if same category, else 1 | unordered categories |
| `binary_sym` | 0 if same, else 1; a shared "No" **counts** as similarity | both values meaningful |
| `binary_asym` | as above, but a pair where **both are "No"** is left out of the comparison | "Yes" marks the presence of an uncommon state |

Fixed ranges (rather than ranges observed in a sample) make distances identical across every sample and in the assignment step.

**Combined distance** between children *i* and *j* (Gower, 1971):

$$ d(i,j) = \frac{\sum_k w_k\, \delta_{ijk}\, d_{ijk}}{\sum_k w_k\, \delta_{ijk}} $$

where $\delta_{ijk}=0$ if feature *k* is missing for either child (only `sleep_below_minimum` for infants < 4 months) or if *k* is asymmetric binary and both children are "No"; otherwise $\delta_{ijk}=1$.

**Exact meanings (2024 NSCH codebook):**
- `blindness`: *"Does this child have … blindness or problems with seeing, even when wearing glasses?"* Described as **vision difficulty (even with glasses)**.
- `k4q23`: *"During the past 12 months, has this child taken any medication because of difficulties with their emotions, concentration, or behavior?"* This is **parent-reported medication use**, not a diagnosis.
- `k10q12–14`: *"In your neighborhood, is/are there"* a park or playground / a recreation or community center or boys' and girls' club / a library or bookmobile (parent report).

**Documented caveats:**
- `sleep_below_minimum` is defined using **age-specific** thresholds while `sc_age_years` is also a feature, so some age-related information overlaps.
- `eyedoctor` is excluded from the primary model ("ever" for T1 vs "past 2 years" for T2/T3) and is tested only in a sensitivity run.
- The original `sleep_hours` codes are not comparable across forms and are not used for clustering.

**Weighting schemes:** W0 gives every feature weight 1. W1 (domain-balanced) gives each domain a total weight of 1, split equally among its features.

**Sensitivity variants** (defined here; run later): `all_symmetric` (every binary symmetric), `with_eyedoctor` (15 features), `no_sleep` (13 features, sleep removed).

In [4]:
FEATURE_TABLE = pd.DataFrame(
    [
        # feature,             domain,        gower_type,   min, max, absent, description
        ("sc_age_years",        "general",     "numeric",     0, 17, np.nan, "Age in years"),
        ("sc_sex",              "general",     "binary_sym",  1,  2, np.nan, "Sex (1 male, 2 female)"),
        ("family_r",            "general",     "nominal",     1,  8, np.nan, "Family structure (8 types)"),
        ("blindness",           "health",      "binary_asym", 1,  2, 2,    "Vision difficulty even with glasses (1 yes, 2 no)"),
        ("k4q22_r",             "health",      "nominal",     1,  3, np.nan, "Mental health professional treatment, past 12 months (yes / no but needed / no, not needed)"),
        ("k4q23",               "health",      "binary_asym", 1,  2, 2,    "Medication for emotions/concentration/behavior, past 12 months (1 yes, 2 no)"),
        ("screentime",          "lifestyle",   "ordinal",     1,  5, np.nan, "Weekday screen time (<1 h ... 4+ h)"),
        ("sleep_below_minimum", "lifestyle",   "binary_sym",  0,  1, np.nan, "Sleep below age-specific recommended minimum (1 yes, 0 no; missing < 4 months)"),
        ("k6q71_r",             "lifestyle",   "ordinal",     1,  4, np.nan, "Shows interest/curiosity in learning (always ... never)"),
        ("higrade",             "environment", "ordinal",     1,  3, np.nan, "Highest adult education (< HS, HS, > HS)"),
        ("foodsit",             "environment", "ordinal",     1,  4, np.nan, "Household food affordability (always nutritious ... often not enough)"),
        ("k10q12",              "environment", "binary_sym",  1,  2, np.nan, "Park or playground in neighborhood"),
        ("k10q13",              "environment", "binary_sym",  1,  2, np.nan, "Recreation/community center or boys' and girls' club in neighborhood"),
        ("k10q14",              "environment", "binary_sym",  1,  2, np.nan, "Library or bookmobile in neighborhood"),
    ],
    columns=["feature", "domain", "gower_type", "min", "max", "absent_code", "description"],
)
EYEDOCTOR_ROW = ("eyedoctor", "health", "binary_sym", 1, 2, np.nan, "Seen an eye doctor (T1: ever; T2/T3: past 2 years)")

# Which imputation flag tracks each feature (features without a flag never had missing values)
FLAG_SOURCE = {"sleep_below_minimum": "sleep_hours"}


def build_spec(variant="primary", weighting="W0"):
    # Return the feature specification (with weights and ranges) for one model variant.
    spec = FEATURE_TABLE.copy()
    if variant == "with_eyedoctor":
        spec = pd.concat([spec, pd.DataFrame([EYEDOCTOR_ROW], columns=spec.columns)], ignore_index=True)
    elif variant == "no_sleep":
        spec = spec[spec["feature"] != "sleep_below_minimum"].reset_index(drop=True)
    elif variant == "all_symmetric":
        spec.loc[spec["gower_type"] == "binary_asym", ["gower_type", "absent_code"]] = ["binary_sym", np.nan]
    elif variant != "primary":
        raise ValueError(variant)

    spec["range"] = spec["max"] - spec["min"]
    if weighting == "W0":
        spec["weight"] = 1.0
    elif weighting == "W1":
        spec["weight"] = 1.0 / spec.groupby("domain")["feature"].transform("count")
    else:
        raise ValueError(weighting)
    return spec


def complete_case_mask(data, spec):
    # True where none of the model's features was imputed. Uses ONLY the model's features' flags;
    # the structural NaN in sleep_below_minimum (infants < 4 months) is not nonresponse and is ignored.
    flags = []
    for f in spec["feature"]:
        flag = f"{FLAG_SOURCE.get(f, f)}_imputed"
        if flag in data.columns:
            flags.append(flag)
    return data[flags].eq(0).all(axis=1), flags


SPEC_PRIMARY = build_spec("primary", "W0")
display(SPEC_PRIMARY.assign(weight_W1=build_spec("primary", "W1")["weight"].round(3)))

,feature,domain,gower_type,min,max,absent_code,description,range,weight,weight_W1
0,sc_age_years,general,numeric,0,17,NaN,Age in years,17,1.0,0.333
1,sc_sex,general,binary_sym,1,2,NaN,"Sex (1 male, 2 female)",1,1.0,0.333
2,family_r,general,nominal,1,8,NaN,Family structure (8 types),7,1.0,0.333
3,blindness,health,binary_asym,1,2,2.0,"Vision difficulty even with glasses (1 yes, 2 no)",1,1.0,0.333
4,k4q22_r,health,nominal,1,3,NaN,"Mental health professional treatment, past 12 ...",2,1.0,0.333
5,k4q23,health,binary_asym,1,2,2.0,Medication for emotions/concentration/behavior...,1,1.0,0.333
6,screentime,lifestyle,ordinal,1,5,NaN,Weekday screen time (<1 h ... 4+ h),4,1.0,0.333
7,sleep_below_minimum,lifestyle,binary_sym,0,1,NaN,Sleep below age-specific recommended minimum (...,1,1.0,0.333
8,k6q71_r,lifestyle,ordinal,1,4,NaN,Shows interest/curiosity in learning (always ....,3,1.0,0.333
9,higrade,environment,ordinal,1,3,NaN,"Highest adult education (< HS, HS, > HS)",2,1.0,0.200


In [5]:
# ---- Validate observed values against the specification ------------------
all_features = pd.concat([FEATURE_TABLE, pd.DataFrame([EYEDOCTOR_ROW], columns=FEATURE_TABLE.columns)])
for row in all_features.itertuples():
    vals = df[row.feature].dropna()
    assert vals.between(row.min, row.max).all(), f"{row.feature}: values outside {row.min}-{row.max}"
    assert (vals % 1 == 0).all(), f"{row.feature}: non-integer codes"
    if row.gower_type.startswith("binary"):
        assert set(vals.unique()) <= {row.min, row.max}, row.feature
    nan_ok = row.feature == "sleep_below_minimum"
    assert nan_ok or df[row.feature].notna().all(), f"{row.feature}: unexpected missing values"
print("All feature values lie within their codebook ranges.")

# ---- Complete-case counts (actual model features only) --------------------
for variant in ["primary", "with_eyedoctor", "no_sleep", "all_symmetric"]:
    spec = build_spec(variant)
    mask, flags = complete_case_mask(df, spec)
    print(f"{variant:15s} {len(spec):2d} features | complete cases: {int(mask.sum()):,} | flags used: {len(flags)}")

cc_primary, _ = complete_case_mask(df, build_spec("primary"))
cc_eye, _ = complete_case_mask(df, build_spec("with_eyedoctor"))
assert int(cc_primary.sum()) == 47_391 and int(cc_eye.sum()) == 47_260
print("\nInfants < 4 months counted as complete (primary):", int((cc_primary & under_4m).sum()), "of", int(under_4m.sum()))

All feature values lie within their codebook ranges.


primary         14 features | complete cases: 47,391 | flags used: 11
with_eyedoctor  15 features | complete cases: 47,260 | flags used: 12
no_sleep        13 features | complete cases: 47,497 | flags used: 10
all_symmetric   14 features | complete cases: 47,391 | flags used: 11

Infants < 4 months counted as complete (primary): 366 of 446


## 3 · Gower distance implementation

No external Gower package is used. The PyPI `gower` package has no ordinal or asymmetric-binary type, and it would treat integer-coded nominal variables as numeric. The implementation below is plain NumPy so every rule above is explicit.

- `gower_cross(XA, XB, spec)`: dissimilarities between every row of `XA` and every row of `XB` (a rectangular block).
- `gower_condensed(X, spec)`: all unique pairs within `X` as a condensed vector, built block by block. This is the format `scipy.cluster.hierarchy.linkage` expects. Memory is n(n−1)/2 × 8 bytes (≈ 381 MiB for n = 10,000), plus one block of temporaries.
- `mean_distance_to_clusters(...)`: for assignment, the **average** Gower distance from each new child to the members of each sample cluster, processed in chunks so a 41,375 × 10,000 matrix is never stored.

**Assignment margin** (used later): for child *x*, $D_c(x) = \frac{1}{|S_c|}\sum_{s\in S_c} d(x,s)$ for each cluster *c* with sample members $S_c$. The child is assigned to the cluster with the smallest $D_c$. Margin = second-smallest $D_c$ − smallest $D_c$, in Gower units (0–1 scale). Exact ties go to the lowest cluster label and are flagged.

**Memory-efficient helpers:**
- `condensed_to_square_float32`: the square matrix needed by silhouette, built without intermediate copies.
- `cophenetic_correlation`: computed in chunks, because scipy's `cophenet(Z, Y)` needs about six condensed-size arrays (about 2.3 GiB at n = 10,000).

**Failure modes guarded against:** a zero denominator (impossible here because most features are never missing or asymmetric, but asserted anyway), NaN leaking into distances, and running out of memory (`require_memory`).

In [6]:
def encode(data, spec):
    # Feature matrix (float64; NaN only for structural missingness), columns in spec order.
    return data[spec["feature"].tolist()].to_numpy(dtype=np.float64)


def gower_cross(XA, XB, spec):
    # Gower dissimilarities between rows of XA (m x p) and rows of XB (n x p) -> (m x n) float64.
    num = np.zeros((XA.shape[0], XB.shape[0]))
    den = np.zeros_like(num)
    for k, f in enumerate(spec.itertuples(index=False)):
        a = XA[:, k][:, None]
        b = XB[:, k][None, :]
        if f.gower_type in ("numeric", "ordinal"):
            d = np.abs(a - b) / f.range
        else:  # nominal, binary_sym, binary_asym: simple matching
            d = (a != b).astype(np.float64)
        valid = ~(np.isnan(a) | np.isnan(b))
        if f.gower_type == "binary_asym":
            valid = valid & ~((a == f.absent_code) & (b == f.absent_code))
        num += f.weight * np.where(valid, d, 0.0)
        den += f.weight * valid
    if not (den > 0).all():
        raise ValueError("Zero Gower denominator: a pair has no comparable features")
    return num / den


def gower_condensed(X, spec, block_rows=GOWER_BLOCK_ROWS):
    # Condensed Gower distance vector (scipy order: (0,1), (0,2), ..., (1,2), ...).
    n = X.shape[0]
    out_bytes = n * (n - 1) // 2 * 8
    block_bytes = block_rows * n * 8 * 6          # num, den, d, valid and temporaries
    require_memory(out_bytes + block_bytes, label=f"gower_condensed n={n}")
    out = np.empty(n * (n - 1) // 2, dtype=np.float64)
    pos = 0
    for start in range(0, n - 1, block_rows):
        stop = min(start + block_rows, n)
        block = gower_cross(X[start:stop], X[start:], spec)   # columns start .. n-1
        for r in range(stop - start):
            row = block[r, r + 1:]                           # j = i+1 .. n-1
            out[pos:pos + row.size] = row
            pos += row.size
    assert pos == out.size
    return out


def mean_distance_to_clusters(X_new, X_ref, label_sets, spec, chunk_rows=ASSIGN_CHUNK_ROWS):
    # For each labelling in label_sets {name: labels of X_ref}, return {name: (clusters, D)} where
    # D[i, c] = average Gower distance from X_new[i] to the reference members of cluster c.
    # The distance block for each chunk is computed once and reused for every labelling.
    require_memory(chunk_rows * X_ref.shape[0] * 8 * 6, label="assignment chunk")
    prepared = {}
    for name, labels in label_sets.items():
        clusters = np.unique(labels)
        onehot = (labels[:, None] == clusters[None, :]).astype(np.float64)
        prepared[name] = (clusters, onehot, onehot.sum(axis=0))
    out = {name: np.empty((X_new.shape[0], len(c))) for name, (c, _, _) in prepared.items()}
    for s in range(0, X_new.shape[0], chunk_rows):
        block = gower_cross(X_new[s:s + chunk_rows], X_ref, spec)
        for name, (clusters, onehot, sizes) in prepared.items():
            out[name][s:s + chunk_rows] = (block @ onehot) / sizes
    return {name: (prepared[name][0], D) for name, D in out.items()}


def condensed_to_square_float32(d):
    # Build the square float32 matrix directly from the condensed vector, one row at a time,
    # avoiding the extra float32 copy of the condensed vector that squareform(d.astype(float32)) makes.
    n = int(round((1 + np.sqrt(1 + 8 * d.size)) / 2))
    require_memory(n * n * 4, label=f"square float32 n={n}")
    D = np.zeros((n, n), dtype=np.float32)
    pos = 0
    for i in range(n - 1):
        row = d[pos:pos + n - 1 - i]
        D[i, i + 1:] = row
        D[i + 1:, i] = row
        pos += row.size
    return D


def cophenetic_correlation(Z, d, chunk=5_000_000):
    # Pearson correlation between cophenetic and Gower distances, computed in chunks.
    # scipy's cophenet(Z, Y) creates ~6 condensed-size temporaries (~2.3 GiB at n = 10,000);
    # this version needs only the cophenetic vector itself plus small chunks.
    c = cophenet(Z)
    mc, md = c.mean(), d.mean()
    sxy = sxx = syy = 0.0
    for s in range(0, d.size, chunk):
        a = c[s:s + chunk] - mc
        b = d[s:s + chunk] - md
        sxy += a @ b
        sxx += a @ a
        syy += b @ b
    return sxy / np.sqrt(sxx * syy)


def assign_from_distances(clusters, D):
    # Nearest cluster by average distance, the margin to the runner-up, and a tie indicator.
    order = np.argsort(D, axis=1, kind="stable")   # stable: exact ties go to the lowest label
    rows = np.arange(D.shape[0])
    d1, d2 = D[rows, order[:, 0]], D[rows, order[:, 1]]
    return pd.DataFrame({
        "cluster": clusters[order[:, 0]],
        "second_best_cluster": clusters[order[:, 1]],
        "assignment_margin": d2 - d1,
        "exact_tie": d2 == d1,
    })

## 4 · Gower validation

The vectorised implementation is checked in four ways before it is used on real samples:

1. **Hand-worked examples:** three constructed children whose distances are calculated by hand (shown below), covering asymmetric exclusion, structural missingness, and both weighting schemes.
2. **Reference implementation:** a deliberately simple, pair-by-pair version that follows the formula literally. It is compared with the vectorised version on all 19,900 pairs of 200 real children, for every model variant and both weighting schemes.
3. **Matrix properties:** symmetry, zero diagonal, values in [0, 1], identical profiles at distance 0, and agreement between the condensed vector and the rectangular block.
4. **Assignment helper:** average distances to clusters agree with a direct calculation.

**Expected output:** every check prints `OK`; any failure stops the notebook.

In [7]:
# ---- 4.1 Hand-worked examples ---------------------------------------------
base = dict(sc_age_years=10, sc_sex=1, family_r=1, blindness=2, k4q22_r=3, k4q23=2, screentime=3,
            sleep_below_minimum=0, k6q71_r=1, higrade=3, foodsit=1, k10q12=1, k10q13=1, k10q14=1,
            eyedoctor=1)
A = dict(base)
B = dict(base, sc_age_years=15, screentime=5, k4q23=1)               # older, more screen time, medication = yes
C = dict(base, sc_age_years=0, sleep_below_minimum=np.nan)           # infant < 4 months (sleep structurally missing)
toy = pd.DataFrame([A, B, C])


def toy_d(i, j, variant="primary", weighting="W0"):
    spec = build_spec(variant, weighting)
    X = encode(toy, spec)
    return gower_cross(X[[i]], X[[j]], spec)[0, 0]


checks = {
    # A vs B, W0: blindness both "No" -> excluded (asymmetric): 13 comparable features.
    #   age 5/17, screen time 2/4, k4q23 1 (A no, B yes)
    "A-B primary W0": (toy_d(0, 1), (5/17 + 2/4 + 1) / 13),
    # Same pair, all-symmetric: blindness now compared (match) -> 14 features.
    "A-B all_symmetric W0": (toy_d(0, 1, "all_symmetric"), (5/17 + 2/4 + 1) / 14),
    # A vs C: sleep missing for C, blindness and k4q23 both "No" -> 11 features; only age differs (10/17).
    "A-C primary W0": (toy_d(0, 2), (10/17) / 11),
    # A vs B, W1: general 1/3 each, health 1/3 each, lifestyle 1/3 each, environment 1/5 each.
    #   denominator = 3(1/3) + 2(1/3) [blindness excluded] + 3(1/3) + 5(1/5) = 11/3
    "A-B primary W1": (toy_d(0, 1, "primary", "W1"), ((1/3) * (5/17) + (1/3) * 1 + (1/3) * (2/4)) / (11/3)),
    # No-sleep variant, A vs C: 13 features minus blindness and k4q23 (both No) = 11; age differs.
    "A-C no_sleep W0": (toy_d(0, 2, "no_sleep"), (10/17) / 11),
    # With eyedoctor (symmetric, both yes), A vs B: 14 comparable features.
    "A-B with_eyedoctor W0": (toy_d(0, 1, "with_eyedoctor"), (5/17 + 2/4 + 1) / 14),
}
for name, (got, want) in checks.items():
    assert np.isclose(got, want, atol=1e-12), (name, got, want)
    print(f"OK  {name:24s} computed {got:.6f} = hand-worked {want:.6f}")

OK  A-B primary W0           computed 0.138009 = hand-worked 0.138009
OK  A-B all_symmetric W0     computed 0.128151 = hand-worked 0.128151
OK  A-C primary W0           computed 0.053476 = hand-worked 0.053476
OK  A-B primary W1           computed 0.163102 = hand-worked 0.163102
OK  A-C no_sleep W0          computed 0.053476 = hand-worked 0.053476
OK  A-B with_eyedoctor W0    computed 0.128151 = hand-worked 0.128151


In [8]:
# ---- 4.2 Reference implementation on 200 real children ---------------------
def gower_pair_reference(x, y, spec):
    # Literal pair-by-pair Gower formula (slow; used only for validation).
    num = den = 0.0
    for k, f in enumerate(spec.itertuples(index=False)):
        a, b = x[k], y[k]
        if np.isnan(a) or np.isnan(b):
            continue
        if f.gower_type == "binary_asym" and a == f.absent_code and b == f.absent_code:
            continue
        d = abs(a - b) / f.range if f.gower_type in ("numeric", "ordinal") else float(a != b)
        num += f.weight * d
        den += f.weight
    return num / den


rng_val = np.random.default_rng(SEED)
special = np.concatenate([
    rng_val.choice(np.flatnonzero(under_4m), 10, replace=False),             # structural NaN in sleep
    rng_val.choice(np.flatnonzero(df["blindness"].eq(1)), 10, replace=False),  # asymmetric "yes"
    rng_val.choice(np.flatnonzero(df["k4q23"].eq(1)), 10, replace=False),      # asymmetric "yes"
])
others = rng_val.choice(np.setdiff1d(np.arange(len(df)), special), 170, replace=False)
val_idx = np.concatenate([special, others])
val = df.iloc[val_idx].reset_index(drop=True)

for variant in ["primary", "all_symmetric", "with_eyedoctor", "no_sleep"]:
    for weighting in ["W0", "W1"]:
        spec = build_spec(variant, weighting)
        X = encode(val, spec)
        fast = gower_condensed(X, spec)
        n = len(X)
        ref = np.array([gower_pair_reference(X[i], X[j], spec) for i in range(n) for j in range(i + 1, n)])
        err = np.max(np.abs(fast - ref))
        assert err < 1e-12, (variant, weighting, err)
        print(f"OK  {variant:15s} {weighting}: {fast.size:,} pairs, max |vectorised - reference| = {err:.1e}")

OK  primary         W0: 19,900 pairs, max |vectorised - reference| = 0.0e+00


OK  primary         W1: 19,900 pairs, max |vectorised - reference| = 0.0e+00


OK  all_symmetric   W0: 19,900 pairs, max |vectorised - reference| = 0.0e+00


OK  all_symmetric   W1: 19,900 pairs, max |vectorised - reference| = 0.0e+00


OK  with_eyedoctor  W0: 19,900 pairs, max |vectorised - reference| = 0.0e+00


OK  with_eyedoctor  W1: 19,900 pairs, max |vectorised - reference| = 0.0e+00


OK  no_sleep        W0: 19,900 pairs, max |vectorised - reference| = 0.0e+00


OK  no_sleep        W1: 19,900 pairs, max |vectorised - reference| = 0.0e+00


In [9]:
# ---- 4.3 Matrix properties and 4.4 assignment helper -----------------------
spec = SPEC_PRIMARY
X = encode(val, spec)
D_full = gower_cross(X, X, spec)
d_cond = gower_condensed(X, spec)

assert np.allclose(D_full, D_full.T, atol=0), "not symmetric"
print("OK  symmetric")
assert np.all(np.diag(D_full) == 0), "non-zero diagonal"
print("OK  zero diagonal (including infants with missing sleep)")
assert D_full.min() >= 0 and D_full.max() <= 1
print(f"OK  all values in [0, 1] (observed {D_full.min():.3f} - {D_full.max():.3f})")
assert np.array_equal(squareform(d_cond, checks=False), D_full - np.diag(np.diag(D_full)))
print("OK  condensed vector matches the full block")

dup = val.duplicated(subset=spec["feature"].tolist(), keep=False)
if dup.any():
    dups = val[dup]
    groups = dups.groupby(spec["feature"].tolist(), dropna=False).indices   # positions within `dups`
    pair = dups.index[next(v for v in groups.values() if len(v) > 1)[:2]]   # map to rows of `val`
    assert D_full[pair[0], pair[1]] == 0
    print(f"OK  identical profiles have distance 0 (rows {pair[0]}, {pair[1]})")

labels_toy = np.where(np.arange(len(X)) < 100, 1, 2)
res = mean_distance_to_clusters(X[:20], X, {"toy": labels_toy}, spec, chunk_rows=7)["toy"]
direct = np.column_stack([D_full[:20, labels_toy == c].mean(axis=1) for c in (1, 2)])
assert np.allclose(res[1], direct, atol=1e-12)
print("OK  chunked average-distance-to-cluster matches direct calculation")
del D_full, d_cond, X

OK  symmetric
OK  zero diagonal (including infants with missing sleep)
OK  all values in [0, 1] (observed 0.000 - 0.875)
OK  condensed vector matches the full block
OK  identical profiles have distance 0 (rows 42, 148)
OK  chunked average-distance-to-cluster matches direct calculation


## 5 · Disjoint stratified replicate samples

**Design (approved):**
- Strata: the 15 combinations of `formtype` (T1/T2/T3) × `screentime` (5 levels), using processed values; 868 children have an imputed screen-time value.
- Each stratum is shuffled once (seed 42) and cut into **5 disjoint parts**. Each part gets the same proportional quota per stratum (largest-remainder rounding to exactly 10,000), so each part reproduces the joint `formtype × screentime` distribution of the processed data **approximately**, within rounding.
- 5 × 10,000 = 50,000 of the 51,375 children are used; the remaining 1,375 belong to no part.
- **Part 0 is the primary clustering sample**; parts 1–4 are used for the stability check. Row order within each part is shuffled (seed 42) so that tie-breaking in the linkage is reproducible.

**Wording and limitations:**
- These are **disjoint stratified replicate samples**, not statistically independent samples: they are drawn without replacement from the same finite dataset.
- Stratification preserves the survey-sample mix of age band and screen time. It does **not** make a sample representative of the U.S. child population (NSCH oversamples ages 0–5 and children with special health care needs; survey weights are not used for sampling).

In [10]:
def stratified_disjoint_parts(data, strata_cols, part_size, n_parts, seed):
    # Return (list of positional-index arrays, quota per stratum). Parts are disjoint and stratified.
    rng = np.random.default_rng(seed)
    groups = data.groupby(strata_cols, sort=True).indices        # {stratum key: positions}
    keys = sorted(groups)
    sizes = pd.Series([len(groups[k]) for k in keys],
                      index=pd.MultiIndex.from_tuples(keys, names=strata_cols))
    exact = sizes * part_size / len(data)
    quota = np.floor(exact).astype(int)
    shortfall = part_size - quota.sum()
    remainders = (exact - quota).sort_values(ascending=False, kind="stable")
    quota.loc[remainders.index[:shortfall]] += 1                   # largest-remainder rounding
    assert quota.sum() == part_size and (quota * n_parts <= sizes).all()

    parts = [[] for _ in range(n_parts)]
    for k in keys:
        shuffled = rng.permutation(groups[k])
        q = int(quota.loc[k])
        for p in range(n_parts):
            parts[p].extend(shuffled[p * q:(p + 1) * q])
    parts = [rng.permutation(np.asarray(p)) for p in parts]        # shuffle row order
    return parts, quota


parts, quota = stratified_disjoint_parts(df, STRATA_COLS, SAMPLE_SIZE, N_PARTS, SEED)

# ---- Validation ------------------------------------------------------------
assert all(len(p) == SAMPLE_SIZE for p in parts)
union = np.concatenate(parts)
assert len(np.unique(union)) == N_PARTS * SAMPLE_SIZE, "parts overlap"
print(f"OK  {N_PARTS} disjoint parts of {SAMPLE_SIZE:,}; {len(df) - len(union):,} children in no part")

full_share = df.groupby(STRATA_COLS).size() / len(df)
strata_check = pd.DataFrame({"full_n": df.groupby(STRATA_COLS).size(), "quota_per_part": quota,
                             "full_%": (full_share * 100).round(2),
                             "part_%": (quota / SAMPLE_SIZE * 100).round(2)})
for p, idx in enumerate(parts):
    counts = df.iloc[idx].groupby(STRATA_COLS).size()
    assert (counts.reindex(quota.index).fillna(0).to_numpy() == quota.to_numpy()).all(), p
print(f"OK  every part matches the stratum quotas exactly; max |part% - full%| = "
      f"{(strata_check['part_%'] - strata_check['full_%']).abs().max():.3f} percentage points")
display(strata_check)

# Balance of NON-stratified characteristics (expected to vary only by chance)
balance = pd.DataFrame({
    (f"part {p}" if p is not None else "full data"): {
        "mean age": d["sc_age_years"].mean(),
        "% vision difficulty": d["blindness"].eq(1).mean() * 100,
        "% medication (k4q23)": d["k4q23"].eq(1).mean() * 100,
        "% sleep below minimum": d["sleep_below_minimum"].mean() * 100,
        "% food: sometimes/often not enough": d["foodsit"].ge(3).mean() * 100,
        "% complete case (primary)": complete_case_mask(d, SPEC_PRIMARY)[0].mean() * 100,
    }
    for p, d in [(None, df)] + [(p, df.iloc[idx]) for p, idx in enumerate(parts)]
}).round(2)
display(balance)

sample_membership = pd.DataFrame({ID_COL: df[ID_COL], "replicate_part": -1})
for p, idx in enumerate(parts):
    sample_membership.loc[idx, "replicate_part"] = p
print(sample_membership["replicate_part"].value_counts().sort_index().to_string())

OK  5 disjoint parts of 10,000; 1,375 children in no part
OK  every part matches the stratum quotas exactly; max |part% - full%| = 0.010 percentage points


full_n  quota_per_part  full_%  part_%
formtype screentime                                        
T1       1.0           4816             937    9.37    9.37
         2.0           3316             646    6.45    6.46
         3.0           3619             704    7.04    7.04
         4.0           1266             246    2.46    2.46
         5.0            824             160    1.60    1.60
T2       1.0           1714             334    3.34    3.34
         2.0           2887             562    5.62    5.62
         3.0           5194            1011   10.11   10.11
         4.0           2867             558    5.58    5.58
         5.0           2912             567    5.67    5.67
T3       1.0            727             142    1.42    1.42
         2.0           1698             331    3.31    3.31
         3.0           5651            1100   11.00   11.00
         4.0           5297            1031   10.31   10.31
         5.0           8587            1671   16.71   16.71

,full data,part 0,part 1,part 2,part 3,part 4
mean age,9.67,9.66,9.67,9.65,9.69,9.66
% vision difficulty,1.42,1.48,1.64,1.18,1.39,1.51
% medication (k4q23),10.42,10.56,10.24,10.36,10.17,10.73
% sleep below minimum,31.44,31.52,31.96,31.38,30.69,31.18
% food: sometimes/often not enough,3.83,3.69,3.84,3.82,3.97,3.81
% complete case (primary),92.25,92.67,92.21,91.96,92.22,92.11


replicate_part
-1     1375
 0    10000
 1    10000
 2    10000
 3    10000
 4    10000


## 6 · Pilot: 2,000 real children (runtime and memory)

**Purpose:** measure actual runtime and memory on real data before committing to n = 10,000. This is a **computational check**: the pilot's cluster results are not interpreted and are not used for any modelling decision.

**Pilot sample:** a stratified subsample of 2,000 children from part 0, using the same 15 strata and seed.

**Steps measured** (primary 14-feature specification):
1. Gower condensed distances (W0), and W1 for comparison.
2. Distance sanity checks: no NaN, range, share of zero distances (identical profiles).
3. Average and complete linkage, with validity and monotonicity checks.
4. Cophenetic correlation.
5. Silhouette (per child) for k = 2–10 under both linkages.
6. Assignment of 1,000 children from part 1 to the pilot clusters, for all k at once.

**Extrapolation to n = 10,000:** distance, linkage and silhouette costs grow with the number of pairs (×25 from 2,000 to 10,000). Assignment cost grows with (children assigned × reference size).

In [11]:
pilot_parts, _ = stratified_disjoint_parts(df.iloc[parts[0]].reset_index(drop=True), STRATA_COLS, PILOT_SIZE, 1, SEED)
pilot_idx = parts[0][pilot_parts[0]]           # positions in df
pilot = df.iloc[pilot_idx].reset_index(drop=True)
assert set(pilot_idx) <= set(parts[0]) and len(pilot) == PILOT_SIZE

spec_w0, spec_w1 = build_spec("primary", "W0"), build_spec("primary", "W1")
X_pilot = encode(pilot, spec_w0)
print("Pilot:", pilot.shape, "| distinct feature profiles:", len(pilot.drop_duplicates(subset=spec_w0["feature"].tolist())))
print("Available RAM before pilot:", f"{available_mib():,.0f} MiB")

with monitor("gower_condensed W0", n=PILOT_SIZE):
    d_w0 = gower_condensed(X_pilot, spec_w0)
with monitor("gower_condensed W1", n=PILOT_SIZE):
    d_w1 = gower_condensed(encode(pilot, spec_w1), spec_w1)

for name, d in [("W0", d_w0), ("W1", d_w1)]:
    assert np.isfinite(d).all() and d.min() >= 0 and d.max() <= 1
    q = np.quantile(d, [0, 0.05, 0.25, 0.5, 0.75, 0.95, 1])
    print(f"{name}: {d.size:,} pairs | zero distances {np.mean(d == 0) * 100:.2f}% | quantiles "
          + " ".join(f"{v:.3f}" for v in q))

Pilot: (2000, 34) | distinct feature profiles: 1783
Available RAM before pilot: 952 MiB


W0: 1,999,000 pairs | zero distances 0.02% | quantiles 0.000 0.145 0.272 0.367 0.465 0.600 0.943
W1: 1,999,000 pairs | zero distances 0.02% | quantiles 0.000 0.154 0.281 0.377 0.475 0.612 0.944


In [12]:
linkages, coph = {}, {}
for method in LINKAGE_METHODS:
    with monitor(f"linkage {method}", n=PILOT_SIZE):
        Z = linkage(d_w0, method=method)
    assert is_valid_linkage(Z) and is_monotonic(Z), method
    with monitor(f"cophenetic {method}", n=PILOT_SIZE):
        coph[method] = cophenetic_correlation(Z, d_w0)
    assert np.isclose(coph[method], cophenet(Z, d_w0)[0], atol=1e-10)   # matches scipy (affordable at n = 2,000)
    linkages[method] = Z
    print(f"OK  {method:8s} linkage valid and monotonic | cophenetic correlation {coph[method]:.3f} (chunked = scipy)")

# Silhouette: one float32 square matrix, reused for all k and both linkages
with monitor("square float32", n=PILOT_SIZE):
    D_sq = condensed_to_square_float32(d_w0)
assert np.array_equal(D_sq, squareform(d_w0.astype(np.float32)))
print("OK  square float32 matrix matches scipy squareform")

pilot_rows = []
with monitor("silhouette k=2..10 x 2 linkages", n=PILOT_SIZE):
    for method, Z in linkages.items():
        for k in K_RANGE:
            labels = fcluster(Z, t=k, criterion="maxclust")
            sizes = np.bincount(labels)[1:]
            s = silhouette_samples(D_sq, labels, metric="precomputed")
            pilot_rows.append({"linkage": method, "k_requested": k, "k_obtained": len(sizes),
                               "mean_silhouette": s.mean(), "smallest_cluster": sizes.min(),
                               "clusters_below_1pct": int((sizes < MIN_CLUSTER_SHARE * PILOT_SIZE).sum()),
                               "singletons": int((sizes == 1).sum())})
del D_sq
pilot_eval = pd.DataFrame(pilot_rows)
print("Pipeline check only (pilot n = 2,000; NOT used for model selection):")
display(pilot_eval.round(3))

OK  average  linkage valid and monotonic | cophenetic correlation 0.649 (chunked = scipy)


OK  complete linkage valid and monotonic | cophenetic correlation 0.497 (chunked = scipy)
OK  square float32 matrix matches scipy squareform


Pipeline check only (pilot n = 2,000; NOT used for model selection):


,linkage,k_requested,k_obtained,mean_silhouette,smallest_cluster,clusters_below_1pct,singletons
0,average,2,2,0.359,2,1,0
1,average,3,3,0.221,2,2,0
2,average,4,4,0.151,2,3,0
3,average,5,5,0.080,2,3,0
4,average,6,6,0.171,2,3,0
5,average,7,7,0.168,2,3,0
6,average,8,8,0.166,2,3,0
7,average,9,9,0.139,2,4,0
8,average,10,10,0.137,2,5,0
9,complete,2,2,0.198,249,0,0


In [13]:
# Assignment timing: 1,000 children from part 1, all k at once (average linkage labels)
assign_idx = parts[1][:ASSIGN_CHUNK_ROWS]
X_assign = encode(df.iloc[assign_idx], spec_w0)
label_sets = {k: fcluster(linkages["average"], t=k, criterion="maxclust") for k in K_RANGE}
with monitor("assignment 1,000 x 2,000 (all k)", n=PILOT_SIZE):
    assigned = mean_distance_to_clusters(X_assign, X_pilot, label_sets, spec_w0)

clusters, Dk = assigned[4]
a4 = assign_from_distances(clusters, Dk)
assert (a4["assignment_margin"] >= 0).all() and np.isfinite(Dk).all()
print("OK  assignment produced finite distances and non-negative margins (example: k = 4)")
print("Margin quantiles (k = 4, pipeline check):", np.round(np.quantile(a4["assignment_margin"], [0, .05, .5, .95]), 4))

OK  assignment produced finite distances and non-negative margins (example: k = 4)
Margin quantiles (k = 4, pipeline check): [0.     0.0099 0.0918 0.2186]


In [14]:
perf = pd.DataFrame(PERF_LOG)
display(perf)

# ---- Extrapolation to n = 10,000 -------------------------------------------
# Runtime: distance, linkage, cophenetic and silhouette work grows with the number of pairs (~25x);
# assignment work grows with (children assigned x reference size).
# Memory is modelled component by component, because only some arrays grow with n^2:
#   C(n)  condensed float64 vector, n(n-1)/2 x 8 bytes      (grows with n^2)
#   S(n)  square float32 matrix, n^2 x 4 bytes                (grows with n^2)
#   Gower block temporaries, GOWER_BLOCK_ROWS x n              (grows linearly with n)
#   Linkage: scipy's nn_chain copies the condensed vector inside compiled code, which tracemalloc
#   cannot see, so the documented size (one extra C) is used instead of the traced value.
def row(step):
    return perf.loc[perf["step"] == step].iloc[0]

n0, n1 = PILOT_SIZE, SAMPLE_SIZE
MiB = 2**20
C = lambda n: n * (n - 1) / 2 * 8 / MiB
S = lambda n: n * n * 4 / MiB
pair_scale = (n1 * (n1 - 1)) / (n0 * (n0 - 1))
assign_scale = (41_375 * n1) / (ASSIGN_CHUNK_ROWS * n0)
gower_block_temps = row("gower_condensed W0")["peak_alloc_mib"] - C(n0)

est = pd.DataFrame([
    ("Gower condensed (per weighting)", row("gower_condensed W0")["seconds"] * pair_scale,
     C(n1) + gower_block_temps * n1 / n0, "C + block temporaries (linear)"),
    ("Average linkage", row("linkage average")["seconds"] * pair_scale, C(n1), "internal copy of C"),
    ("Complete linkage", row("linkage complete")["seconds"] * pair_scale, C(n1), "internal copy of C"),
    ("Cophenetic correlation (per linkage)", row("cophenetic average")["seconds"] * pair_scale,
     C(n1) + 2 * 5_000_000 * 8 / MiB, "cophenetic vector + 2 chunks"),
    ("Square float32 for silhouette", row("square float32")["seconds"] * pair_scale, S(n1), "S"),
    ("Silhouette k=2..10 x 2 linkages", row("silhouette k=2..10 x 2 linkages")["seconds"] * pair_scale,
     256, "scikit-learn working-memory cap"),
    ("Assignment of 41,375 children (all k)", row("assignment 1,000 x 2,000 (all k)")["seconds"] * assign_scale,
     row("assignment 1,000 x 2,000 (all k)")["peak_alloc_mib"] * n1 / n0, "chunk 1,000 x n (linear)"),
], columns=["step", "est_seconds_n10k", "est_step_alloc_mib_n10k", "memory model"]).round(1)
display(est)

baseline = rss_mib()
# Largest simultaneous holding in the planned order: d (C) is kept while linkage copies it (C) and while
# the cophenetic vector (C) exists; d and the square matrix (S) coexist briefly before d is deleted.
peak_concurrent = max(2 * C(n1) + 76, C(n1) + S(n1))      # 76 MiB = two cophenetic chunks
per_run_seconds = est["est_seconds_n10k"].sum() + est.loc[3, "est_seconds_n10k"]   # cophenetic x 2 linkages
print(f"Exact sizes at n = 10,000: condensed float64 C = {C(n1):,.0f} MiB; square float32 S = {S(n1):,.0f} MiB")
print(f"Estimated peak of simultaneously held arrays: {peak_concurrent:,.0f} MiB "
      f"+ process baseline {baseline:,.0f} MiB = {peak_concurrent + baseline:,.0f} MiB")
print(f"Estimated time per full primary run (both linkages, all k, assignment): {per_run_seconds / 60:.1f} min")
print(f"Process peak working set during this notebook: {PROC.memory_info().peak_wset / MiB:,.0f} MiB")
print(f"RAM available now: {available_mib():,.0f} MiB")

,step,n,seconds,peak_alloc_mib,rss_before_mib,rss_after_mib,available_after_mib
0,gower_condensed W0,2000,0.904,54.5,193.0,209.2,904.0
1,gower_condensed W1,2000,0.897,54.6,209.2,224.5,866.0
2,linkage average,2000,0.100,1.9,224.5,224.6,853.0
3,cophenetic average,2000,0.087,45.8,224.6,224.6,850.0
4,linkage complete,2000,0.095,2.1,224.6,224.6,852.0
5,cophenetic complete,2000,0.090,45.9,224.6,224.6,845.0
6,square float32,2000,0.060,15.4,224.6,239.9,812.0
7,silhouette k=2..10 x 2 linkages,2000,2.374,0.5,239.9,240.1,771.0
8,"assignment 1,000 x 2,000 (all k)",2000,0.701,79.5,224.8,224.8,722.0


,step,est_seconds_n10k,est_step_alloc_mib_n10k,memory model
0,Gower condensed (per weighting),22.6,577.7,C + block temporaries (linear)
1,Average linkage,2.5,381.4,internal copy of C
2,Complete linkage,2.4,381.4,internal copy of C
3,Cophenetic correlation (per linkage),2.2,457.7,cophenetic vector + 2 chunks
4,Square float32 for silhouette,1.5,381.5,S
5,Silhouette k=2..10 x 2 linkages,59.4,256.0,scikit-learn working-memory cap
6,"Assignment of 41,375 children (all k)",145.0,397.5,"chunk 1,000 x n (linear)"


Exact sizes at n = 10,000: condensed float64 C = 381 MiB; square float32 S = 381 MiB
Estimated peak of simultaneously held arrays: 839 MiB + process baseline 225 MiB = 1,064 MiB
Estimated time per full primary run (both linkages, all k, assignment): 4.0 min
Process peak working set during this notebook: 316 MiB
RAM available now: 722 MiB
